In [1]:
import lightgbm as lgb

from lightgbm import LGBMClassifier

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix
)
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
import optuna

In [2]:
train_df = pd.read_csv("training_gom(2003-2023).csv")
test_df = pd.read_csv("test_gom(2024-25).csv")

print(train_df.shape)
print(test_df.shape)

(4082861, 14)
(273283, 14)


In [3]:
FEATURES = [
    "CHLOR_A",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos",
    "LAT_scaled",
    "LON_scaled"
]

TARGET = "PHYTOBLOOM"

In [4]:
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

In [5]:
lgb_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    random_state=42,
)

lgb_model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027310 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 928
[LightGBM] [Info] Number of data points in the train set: 4082861, number of used features: 7
[LightGBM] [Info] Start training from score -0.145816
[LightGBM] [Info] Start training from score -3.131013
[LightGBM] [Info] Start training from score -2.385858


,objective,'multiclass'
,random_state,42
,num_class,3
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,class_weight,None
,min_split_gain,0.0


In [6]:
train_pred = lgb_model.predict(X_train)
test_pred = lgb_model.predict(X_test)

In [7]:
print("TRAIN RESULTS\n")

print(classification_report(y_train, train_pred))


print("Macro F1:",
      f1_score(y_train, train_pred, average='macro'))


print("\n\nTEST RESULTS\n")

print(classification_report(y_test, test_pred))


print("Macro F1:",
      f1_score(y_test, test_pred, average='macro'))

TRAIN RESULTS

              precision    recall  f1-score   support

         0.0       0.98      0.99      0.99   3528884
         1.0       0.81      0.69      0.75    178313
         2.0       0.88      0.87      0.87    375664

    accuracy                           0.97   4082861
   macro avg       0.89      0.85      0.87   4082861
weighted avg       0.96      0.97      0.96   4082861

Macro F1: 0.8672997732201653


TEST RESULTS

              precision    recall  f1-score   support

         0.0       0.98      0.98      0.98    232342
         1.0       0.66      0.72      0.69     12506
         2.0       0.86      0.80      0.83     28435

    accuracy                           0.95    273283
   macro avg       0.83      0.83      0.83    273283
weighted avg       0.95      0.95      0.95    273283

Macro F1: 0.8327760553611645


In [8]:
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": lgb_model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance)

      Feature  Importance
5  LAT_scaled        2361
6  LON_scaled        2330
0     CHLOR_A        1646
1     day_sin        1277
2     day_cos        1162
4   month_cos         117
3   month_sin         107


In [12]:
from sklearn.utils.class_weight import compute_sample_weight
def objective_bob_v2(trial):
    params = {
        "objective": "multiclass",
        "num_class": 3,
        "metric": "multi_logloss",

        "n_estimators": trial.suggest_int("n_estimators", 20, 350, step=10),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.20, log=True),

        # Capped down from 13 — that depth is likely what caused the
        # CV-vs-final-fit mismatch
        "max_depth": trial.suggest_int("max_depth", 4, 14),

        # Capped down from 287 for the same reason
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),

        # Keep the low floor — minority classes still need small leaves
        # to get captured, just don't let overall depth run away
        "min_child_samples": trial.suggest_int("min_child_samples", 8, 50),

        "subsample": trial.suggest_float("subsample", 0.65, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.95),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 1e-3, 0.5, log=True),

        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }

    weight_mode = trial.suggest_categorical("weight_mode", ["none", "balanced", "sqrt_balanced"])

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gaps, val_scores = [], []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        if weight_mode == "none":
            sw = None
        elif weight_mode == "balanced":
            sw = compute_sample_weight(class_weight="balanced", y=y_tr)
        else:
            raw_w = compute_sample_weight(class_weight="balanced", y=y_tr)
            sw = np.sqrt(raw_w)

        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr, sample_weight=sw)

        train_f1 = f1_score(y_tr, model.predict(X_tr), average="macro")
        val_f1 = f1_score(y_val, model.predict(X_val), average="macro")

        val_scores.append(val_f1)
        gaps.append(train_f1 - val_f1)

    mean_val = np.mean(val_scores)
    mean_gap = np.mean(gaps)

    return mean_val, mean_gap

In [13]:
study_bob_v2 = optuna.create_study(directions=["maximize", "minimize"]) 
study_bob_v2.optimize(objective_bob_v2, n_trials=60, show_progress_bar=True)

[I 2026-07-21 10:06:07,847] A new study created in memory with name: no-name-8fe83719-ce41-489f-b02d-bbc501a5720a


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-07-21 10:10:22,214] Trial 0 finished with values: [0.8341847046939442, 0.00031374970529631787] and parameters: {'n_estimators': 190, 'learning_rate': 0.045455890910509236, 'max_depth': 4, 'num_leaves': 148, 'min_child_samples': 32, 'subsample': 0.7484038067687234, 'colsample_bytree': 0.9379648387638477, 'reg_alpha': 0.0038747435036323237, 'reg_lambda': 0.004739109814655123, 'min_split_gain': 0.10955164469605104, 'weight_mode': 'sqrt_balanced'}.
[I 2026-07-21 10:12:08,337] Trial 1 finished with values: [0.8659994604749114, 0.0012534109704369634] and parameters: {'n_estimators': 60, 'learning_rate': 0.10751971540754268, 'max_depth': 7, 'num_leaves': 131, 'min_child_samples': 11, 'subsample': 0.7665903393173361, 'colsample_bytree': 0.8478814226832868, 'reg_alpha': 0.0010796903468967495, 'reg_lambda': 0.004575338481423659, 'min_split_gain': 0.008406501368059751, 'weight_mode': 'none'}.
[I 2026-07-21 10:22:46,456] Trial 2 finished with values: [0.8013943993983303, 0.0012343475558215

In [14]:
best_trials = study_bob_v2.best_trials
for t in best_trials:
    print(f"macro_f1={t.values[0]:.4f}, gap={t.values[1]:.4f}, params={t.params}")

macro_f1=0.8781, gap=0.0043, params={'n_estimators': 270, 'learning_rate': 0.09672391431477959, 'max_depth': 7, 'num_leaves': 76, 'min_child_samples': 48, 'subsample': 0.919087735645794, 'colsample_bytree': 0.797078529988762, 'reg_alpha': 0.7669763639758024, 'reg_lambda': 0.2075721585963877, 'min_split_gain': 0.011081347917872747, 'weight_mode': 'none'}
macro_f1=0.8415, gap=0.0005, params={'n_estimators': 160, 'learning_rate': 0.02548637275603568, 'max_depth': 6, 'num_leaves': 79, 'min_child_samples': 28, 'subsample': 0.8967247186272956, 'colsample_bytree': 0.6777555090327676, 'reg_alpha': 0.09216896136931316, 'reg_lambda': 0.005643018577923217, 'min_split_gain': 0.05754734232334371, 'weight_mode': 'sqrt_balanced'}
macro_f1=0.8713, gap=0.0021, params={'n_estimators': 80, 'learning_rate': 0.06370462636454753, 'max_depth': 11, 'num_leaves': 125, 'min_child_samples': 27, 'subsample': 0.9044385219583599, 'colsample_bytree': 0.8840729740078569, 'reg_alpha': 0.12014890795115367, 'reg_lambda'

In [36]:
from sklearn.utils.class_weight import compute_sample_weight
def objective_bob_v2(trial):
    params = {
        "objective": "multiclass",
        "num_class": 3,
        "metric": "multi_logloss",

        "n_estimators": trial.suggest_int("n_estimators", 10, 120, step=5),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.20, log=True),

        # Capped down from 13 — that depth is likely what caused the
        # CV-vs-final-fit mismatch
        "max_depth": trial.suggest_int("max_depth", 4, 12),

        # Capped down from 287 for the same reason
        "num_leaves": trial.suggest_int("num_leaves", 30, 150),

        # Keep the low floor — minority classes still need small leaves
        # to get captured, just don't let overall depth run away
        "min_child_samples": trial.suggest_int("min_child_samples", 8, 50),

        "subsample": trial.suggest_float("subsample", 0.65, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.95),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 1e-3, 0.5, log=True),

        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }

    weight_mode = trial.suggest_categorical("weight_mode", ["none", "balanced", "sqrt_balanced"])

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gaps, val_scores = [], []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        if weight_mode == "none":
            sw = None
        elif weight_mode == "balanced":
            sw = compute_sample_weight(class_weight="balanced", y=y_tr)
        else:
            raw_w = compute_sample_weight(class_weight="balanced", y=y_tr)
            sw = np.sqrt(raw_w)

        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr, sample_weight=sw)

        train_f1 = f1_score(y_tr, model.predict(X_tr), average="macro")
        val_f1 = f1_score(y_val, model.predict(X_val), average="macro")

        val_scores.append(val_f1)
        gaps.append(train_f1 - val_f1)

    mean_val = np.mean(val_scores)
    mean_gap = np.mean(gaps)

    return mean_val, mean_gap

In [37]:
study = optuna.create_study(directions=["maximize", "minimize"]) 
study.optimize(objective_bob_v2, n_trials=40, show_progress_bar=True)

[I 2026-07-22 10:18:45,492] A new study created in memory with name: no-name-729a283f-afe7-4325-ac9b-8b6ccfcf8203


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-07-22 10:19:03,658] Trial 0 finished with values: [0.8236441742692149, 0.0004201954106122452] and parameters: {'n_estimators': 10, 'learning_rate': 0.1018018174502167, 'max_depth': 12, 'num_leaves': 36, 'min_child_samples': 18, 'subsample': 0.7886220714445638, 'colsample_bytree': 0.8833799506657071, 'reg_alpha': 1.2267084731584215, 'reg_lambda': 0.022871086282034124, 'min_split_gain': 0.24671441641567377, 'weight_mode': 'none'}.
[I 2026-07-22 10:19:58,505] Trial 1 finished with values: [0.8415925389790451, 0.00047854180133490856] and parameters: {'n_estimators': 40, 'learning_rate': 0.037381170358262945, 'max_depth': 8, 'num_leaves': 39, 'min_child_samples': 8, 'subsample': 0.8512635538854962, 'colsample_bytree': 0.825522750882135, 'reg_alpha': 0.0018314926404018692, 'reg_lambda': 0.8051024545864424, 'min_split_gain': 0.09400510213250773, 'weight_mode': 'none'}.
[I 2026-07-22 10:21:08,816] Trial 2 finished with values: [0.8368235258421318, 0.0003458097698397555] and parameters:

In [38]:
best_trials = study.best_trials
for t in best_trials:
    print(f"macro_f1={t.values[0]:.4f}, gap={t.values[1]:.4f}, params={t.params}")

macro_f1=0.8241, gap=0.0002, params={'n_estimators': 30, 'learning_rate': 0.15186102650037084, 'max_depth': 4, 'num_leaves': 47, 'min_child_samples': 10, 'subsample': 0.8555916134419175, 'colsample_bytree': 0.9389465016059346, 'reg_alpha': 0.004887214402994542, 'reg_lambda': 0.0037178332507581072, 'min_split_gain': 0.0034824732900108624, 'weight_mode': 'sqrt_balanced'}
macro_f1=0.8667, gap=0.0015, params={'n_estimators': 65, 'learning_rate': 0.0872444857727788, 'max_depth': 8, 'num_leaves': 106, 'min_child_samples': 39, 'subsample': 0.8629488931365125, 'colsample_bytree': 0.6829976908131254, 'reg_alpha': 4.986158430142343, 'reg_lambda': 0.013007175487204793, 'min_split_gain': 0.0015048248308524189, 'weight_mode': 'none'}
macro_f1=0.8715, gap=0.0020, params={'n_estimators': 85, 'learning_rate': 0.12983751244292685, 'max_depth': 11, 'num_leaves': 58, 'min_child_samples': 9, 'subsample': 0.7814521287283342, 'colsample_bytree': 0.7983676255166985, 'reg_alpha': 2.694575590102847, 'reg_lambd

In [39]:
i=0
for t in best_trials:
    params=t.params
    lgb_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    **params,
    random_state=42
    )

    lgb_model.fit(X_train, y_train)
    y_pred1 = lgb_model.predict(
        X_train
    )

    y_pred = lgb_model.predict(
        X_test
    )
    macro_f1_train = f1_score(
        y_train,
        y_pred1,
        average="macro"
    )

    macro_f1_test = f1_score(
        y_test,
        y_pred,
        average="macro"
    )
    gap=macro_f1_train-macro_f1_test
    print(f"{i} Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")
    i+=1


0 Gap: 0.026407961506596855 Train Macro F1 : 0.8388840100171131  Test Macro F1 : 0.8124760485105162
1 Gap: 0.036245489587266255 Train Macro F1 : 0.8684261439055261  Test Macro F1 : 0.8321806543182598
2 Gap: 0.04096700430458278 Train Macro F1 : 0.8737820247974281  Test Macro F1 : 0.8328150204928453
3 Gap: 0.026456946685845195 Train Macro F1 : 0.836972229158936  Test Macro F1 : 0.8105152824730908
4 Gap: 0.03486525871636814 Train Macro F1 : 0.8658124555088896  Test Macro F1 : 0.8309471967925215
5 Gap: 0.03147158857609489 Train Macro F1 : 0.8588979952560472  Test Macro F1 : 0.8274264066799523
6 Gap: 0.0027306125088594757 Train Macro F1 : 0.3090735873749493  Test Macro F1 : 0.3063429748660898
7 Gap: 0.030305337336218763 Train Macro F1 : 0.854596137635078  Test Macro F1 : 0.8242908002988593
8 Gap: 0.02693152125153009 Train Macro F1 : 0.8473275715078538  Test Macro F1 : 0.8203960502563237
9 Gap: 0.038200429605810715 Train Macro F1 : 0.8695209266126693  Test Macro F1 : 0.8313204970068586
10 Ga

In [45]:
params=best_trials[12].params
params

{'n_estimators': 60,
 'learning_rate': 0.02738388850765012,
 'max_depth': 12,
 'num_leaves': 39,
 'min_child_samples': 46,
 'subsample': 0.8704588059140501,
 'colsample_bytree': 0.8013089285041655,
 'reg_alpha': 0.06198542335852684,
 'reg_lambda': 0.004700171913470125,
 'min_split_gain': 0.0012963994780004645,
 'weight_mode': 'sqrt_balanced'}

In [46]:
lgb_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    **params,
    random_state=42
)

lgb_model.fit(X_train, y_train)
y_pred1 = lgb_model.predict(
    X_train
)

y_pred = lgb_model.predict(
    X_test
)
macro_f1_train = f1_score(
    y_train,
    y_pred1,
    average="macro"
)

macro_f1_test = f1_score(
    y_test,
    y_pred,
    average="macro"
)
gap=macro_f1_train-macro_f1_test
print(f" Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")

 Gap: 0.023233052630644124 Train Macro F1 : 0.8455972920721865  Test Macro F1 : 0.8223642394415424


In [47]:
import joblib

joblib.dump(lgb_model, 'lgb_model_GOM.pkl')

['lgb_model_GOM.pkl']

In [20]:
study = optuna.create_study(directions=["maximize", "minimize"]) 
study.optimize(objective_bob_v2, n_trials=60, show_progress_bar=True)

[I 2026-07-21 22:30:47,677] A new study created in memory with name: no-name-2d9b38f4-d778-4bd2-bfd6-bf0869c730e6


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-07-21 22:32:22,318] Trial 0 finished with values: [0.8120269670085559, 0.002378080435606789] and parameters: {'n_estimators': 55, 'learning_rate': 0.1340093372711239, 'max_depth': 12, 'num_leaves': 124, 'min_child_samples': 11, 'subsample': 0.7209371052273508, 'colsample_bytree': 0.9187432722773693, 'reg_alpha': 0.0025129250708320577, 'reg_lambda': 1.1978194110833082, 'min_split_gain': 0.035588985615435886, 'weight_mode': 'balanced'}.
[I 2026-07-21 22:33:55,874] Trial 1 finished with values: [0.8693709118493368, 0.0015271048057487802] and parameters: {'n_estimators': 70, 'learning_rate': 0.18647638505295122, 'max_depth': 8, 'num_leaves': 44, 'min_child_samples': 36, 'subsample': 0.9275427040003066, 'colsample_bytree': 0.8581952249965697, 'reg_alpha': 4.107729394590023, 'reg_lambda': 0.16887893583053262, 'min_split_gain': 0.00937981899470357, 'weight_mode': 'none'}.
[I 2026-07-21 22:37:56,151] Trial 2 finished with values: [0.8095557699617503, 0.002257121686214214] and parameter

In [21]:
best_trials = study.best_trials
for t in best_trials:
    print(f"macro_f1={t.values[0]:.4f}, gap={t.values[1]:.4f}, params={t.params}")

macro_f1=0.8694, gap=0.0015, params={'n_estimators': 70, 'learning_rate': 0.18647638505295122, 'max_depth': 8, 'num_leaves': 44, 'min_child_samples': 36, 'subsample': 0.9275427040003066, 'colsample_bytree': 0.8581952249965697, 'reg_alpha': 4.107729394590023, 'reg_lambda': 0.16887893583053262, 'min_split_gain': 0.00937981899470357, 'weight_mode': 'none'}
macro_f1=0.8758, gap=0.0030, params={'n_estimators': 105, 'learning_rate': 0.0836916450326031, 'max_depth': 14, 'num_leaves': 113, 'min_child_samples': 36, 'subsample': 0.7155935128576547, 'colsample_bytree': 0.8026350537096177, 'reg_alpha': 0.5340886256132067, 'reg_lambda': 0.07969069468228744, 'min_split_gain': 0.015891342498951612, 'weight_mode': 'none'}
macro_f1=0.7220, gap=0.0001, params={'n_estimators': 30, 'learning_rate': 0.0553206305102532, 'max_depth': 4, 'num_leaves': 31, 'min_child_samples': 14, 'subsample': 0.781125970247935, 'colsample_bytree': 0.7336494404549536, 'reg_alpha': 0.12537600438004842, 'reg_lambda': 0.003330045

In [22]:
i=0
for t in best_trials:
    params=t.params
    lgb_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    **params,
    random_state=42
    )

    lgb_model.fit(X_train, y_train)
    y_pred1 = lgb_model.predict(
        X_train
    )

    y_pred = lgb_model.predict(
        X_test
    )
    macro_f1_train = f1_score(
        y_train,
        y_pred1,
        average="macro"
    )

    macro_f1_test = f1_score(
        y_test,
        y_pred,
        average="macro"
    )
    gap=macro_f1_train-macro_f1_test
    print(f"{i} Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")
    i+=1


0 Gap: 0.03850995854378414 Train Macro F1 : 0.8714439602745836  Test Macro F1 : 0.8329340017307995
1 Gap: 0.0437538617215778 Train Macro F1 : 0.8790023671573902  Test Macro F1 : 0.8352485054358124
2 Gap: 0.022084422593229736 Train Macro F1 : 0.804083033748315  Test Macro F1 : 0.7819986111550853
3 Gap: 0.035762026375271705 Train Macro F1 : 0.8681129523855646  Test Macro F1 : 0.8323509260102929
4 Gap: 0.03376843394672058 Train Macro F1 : 0.8637329930605094  Test Macro F1 : 0.8299645591137889
5 Gap: 0.041713617694255745 Train Macro F1 : 0.8761406123523886  Test Macro F1 : 0.8344269946581329
6 Gap: 0.022667832065766058 Train Macro F1 : 0.8129300404386849  Test Macro F1 : 0.7902622083729188
7 Gap: 0.0027306125088594757 Train Macro F1 : 0.3090735873749493  Test Macro F1 : 0.3063429748660898
8 Gap: 0.024817647278929544 Train Macro F1 : 0.8472942049376035  Test Macro F1 : 0.822476557658674
9 Gap: 0.02567454019400839 Train Macro F1 : 0.8332828658054391  Test Macro F1 : 0.8076083256114307
10 Gap

In [34]:
best_trials[13].params

{'n_estimators': 100,
 'learning_rate': 0.023508080592633908,
 'max_depth': 7,
 'num_leaves': 37,
 'min_child_samples': 27,
 'subsample': 0.7953533086600533,
 'colsample_bytree': 0.8481859655309312,
 'reg_alpha': 3.4747080510246486,
 'reg_lambda': 0.15569123934505041,
 'min_split_gain': 0.010061202362508704,
 'weight_mode': 'none'}

In [15]:
i=0
for t in best_trials:
    params=t.params
    lgb_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    **params,
    random_state=42
    )

    lgb_model.fit(X_train, y_train)
    y_pred1 = lgb_model.predict(
        X_train
    )

    y_pred = lgb_model.predict(
        X_test
    )
    macro_f1_train = f1_score(
        y_train,
        y_pred1,
        average="macro"
    )

    macro_f1_test = f1_score(
        y_test,
        y_pred,
        average="macro"
    )
    gap=macro_f1_train-macro_f1_test
    print(f"{i} Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")
    i+=1


0 Gap: 0.04846896465967776 Train Macro F1 : 0.8824297045533142  Test Macro F1 : 0.8339607398936364
1 Gap: 0.0313512456777284 Train Macro F1 : 0.8564225960807775  Test Macro F1 : 0.8250713504030491
2 Gap: 0.03946386414628289 Train Macro F1 : 0.873452346075902  Test Macro F1 : 0.8339884819296192
3 Gap: 0.035664671249273416 Train Macro F1 : 0.868188179256545  Test Macro F1 : 0.8325235080072716
4 Gap: 0.02319056369240069 Train Macro F1 : 0.8421942730824594  Test Macro F1 : 0.8190037093900587
5 Gap: 0.041046371534525505 Train Macro F1 : 0.8731956367262036  Test Macro F1 : 0.8321492651916781
6 Gap: 0.0683648525106656 Train Macro F1 : 0.8994470941751588  Test Macro F1 : 0.8310822416644932
7 Gap: 0.03451234677233472 Train Macro F1 : 0.8659881915499129  Test Macro F1 : 0.8314758447775782
8 Gap: 0.0692187846489618 Train Macro F1 : 0.900489638445738  Test Macro F1 : 0.8312708537967762
9 Gap: 0.03154960254811967 Train Macro F1 : 0.8611369946720299  Test Macro F1 : 0.8295873921239102
10 Gap: 0.0478

In [17]:
best_trials[4].params

{'n_estimators': 120,
 'learning_rate': 0.022961788844571184,
 'max_depth': 5,
 'num_leaves': 107,
 'min_child_samples': 17,
 'subsample': 0.74006228293688,
 'colsample_bytree': 0.8890092158285131,
 'reg_alpha': 0.04211243983254,
 'reg_lambda': 0.00159421307650509,
 'min_split_gain': 0.01783762149547091,
 'weight_mode': 'none'}